In [1]:
import sys, os
import pickle as pk
import numpy as np
from itertools import product
import matplotlib.pyplot as plt
from rich import print as rprint
from multiprocessing import Pool

from utils import *

sys.path.append("../")
from sangrah.tools import get_info

sys.path.append('../../scripts_analysis')
from analysis import *

In [2]:
dataPath = "../../data"
resultPath = os.path.join(dataPath, 'control')
tstep = 0.1 #ms

In [3]:
category_info_path = os.path.join(dataPath, 'control_eachCa/category_info.pk')
if os.path.exists(category_info_path):
    with open(category_info_path, "rb") as infile:
        category_info = pk.load(infile)

# rprint(category_info[0,0,0,0]['rep_design_median'])

In [4]:
categories = []
for category in category_info.ravel():
    # rprint(category)
    if category:
        categories.append(str(category['rep_design_median']))

# len(categories), categories

In [5]:
def run_all(design, tstep=tstep):
    resultDir = os.path.join(resultPath, design)
    simInfo = get_info(design, skip=0)
    ISI, nAP, RRP = int(simInfo['ISI']), int(simInfo['nAP']), simInfo['RRP']
    # print(design, RRP)

    inf = os.path.join(resultDir, 'vesData.dat')
    with open(inf, "rb") as infile:
        vesData = pk.load(infile)
        # print(vesData.keys())
        
    vrel_times = [[(1+float(val))*1000 for val in vals] for vals in vesData[RRP]]
    # vrel_times = vrel_times[:50] ##### For testing

    nMFB, spid, sptimes = get_spike_generator_info(vrel_times)
    # print(nMFB, np.array(spid), np.array(sptimes))

    # print(f"Running simulation for: {design}")
    stMCA3p, spMCA3p = run_sim(nMFB, spid, sptimes, tstep=tstep)
    # test_plot(stMCA3p, spMCA3p, vrel_times)

    ## Spike times of CA3p
    CA3p_spikes = [np.sort([t-1000 for t in (a/ms).tolist() if t>1000]) for a in 
                   spMCA3p.spike_trains().values()]
    
    ## Synaptic currents
    epscs = -stMCA3p.Isyn[:,int(1000/tstep):]/uA

    ## timepoints
    tt = stMCA3p.t[int(1000/tstep):]/ms - 1000

    ## Save the results
    out = os.path.join(resultDir, 'CA3p.pk')
    with open(out, "wb") as outfile:
        pk.dump([CA3p_spikes, epscs, tt], outfile)

    return None
    # return stMCA3p, spMCA3p


In [6]:
# for design in categories[:1]:
#     run_all(design)

In [7]:
with Pool(20) as p:
    p.map(run_all, categories)

In [8]:
# stMCA3p, spMCA3p = run_all(categories[0])

# # epscs = -stMCA3p.Isyn[:,int(1000/tstep):]/uA
# # print(epscs.shape)

# ## Get list of spike times (ms) of CA3p neurons
# CA3p_spikes = [np.sort([t-1000 for t in (a/ms).tolist() if t>1000]) for a in 
#                spMCA3p.spike_trains().values()]
# # print(CA3p_spikes)

# tt = stMCA3p.t[int(1000/tstep):]/ms - 1000
# # print(tt)

# plt.plot(tt, np.mean(epscs, axis=0))